# 🟡 بيقش — تدريب نموذج التعرف على العملة

دفتر جاهز يشغّل خط المعالجة المُصَحَّح كاملاً: تكثير بلا تسريب ← تدريب بهندسة موحّدة ← تصدير `bigash_v18.tflite`.

**مهم:** من قائمة `Runtime` ← `Change runtime type` اختر **GPU** قبل البدء، ثم شغّل الخلايا بالترتيب من الأعلى للأسفل.

## ١) التحقق من البيئة

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else 'لا يوجد GPU — افتح Runtime > Change runtime type > GPU')

## ٢) ربط Drive واستخراج الأدوات من مجلد `scripts/`

بنية مشروعك على Drive: `datasets/` (البيانات مضغوطة) · `scripts/` (الأدوات مضغوطة) · `runs/` (المخرجات).
الخلية التالية تربط Drive، تكتشف مجلد المشروع تلقائياً، وتستخرج أحدث ملف zip من `scripts/` إلى `tools/`.

⚠️ تأكّد أن ملف الأدوات المضغوط في `scripts/` يحوي **النسخة المُحدَّثة**:
`augment_dataset.py` (المتوازن) و `train_bigash_model.py` و **`eval_model.py`** الجديد.

In [ ]:
import os, glob, zipfile
from google.colab import drive
drive.mount('/content/drive')

# ✏️ مجلد مشروعك على Drive (الذي يحوي datasets/ scripts/ runs/).
# اتركه '' للاكتشاف التلقائي، أو ضعه يدوياً مثل '/content/drive/MyDrive/bigash'.
BASE = ''
if not BASE:
    for root, dirs, _ in os.walk('/content/drive/MyDrive'):
        if {'datasets', 'scripts', 'runs'} <= set(dirs):
            BASE = root
            break
    if not BASE:
        BASE = '/content/drive/MyDrive'
print('BASE =', BASE)
for d in ('datasets', 'scripts'):
    assert os.path.isdir(os.path.join(BASE, d)), f'لم أجد {d}/ داخل BASE — عدّل BASE يدوياً.'

# استخراج أحدث zip من scripts/ إلى tools/
os.makedirs('tools', exist_ok=True)
szips = sorted(glob.glob(os.path.join(BASE, 'scripts', '*.zip')), key=os.path.getmtime)
assert szips, 'لا يوجد ملف zip في scripts/.'
print('أدوات من:', szips[-1])
with zipfile.ZipFile(szips[-1]) as z:
    z.extractall('/content/_tools_zip')
for p in glob.glob('/content/_tools_zip/**/*.py', recursive=True):
    os.replace(p, os.path.join('tools', os.path.basename(p)))
print('tools:', sorted(os.listdir('tools')))

# تحقّق أن المرفوع هو النسخة المُحدَّثة (المتوازنة + سكربت التقييم).
with open('tools/augment_dataset.py', encoding='utf-8') as f:
    _aug = f.read()
assert 'target_per_class' in _aug, \
    'نسخة قديمة من augment_dataset.py! أعد ضغط tools المُحدَّث (الوضع المتوازن).'
assert os.path.exists('tools/eval_model.py'), \
    'eval_model.py مفقود — أعد ضغط مجلد tools المُحدَّث بالكامل.'
print('✅ الأدوات جاهزة (المتوازنة + eval_model.py).')

## ٣) استخراج البيانات من مجلد `datasets/`

الخلية التالية تستخرج أحدث ملف zip من `datasets/` وتكتشف مجلد الفئات تلقائياً.
يجب أن يحوي مجلداً فرعياً لكل فئة (٩ فئات):

```
other/  y50/  y100_new/  y100_old/  y200_new/  y200_old/  y250/  y500/  y1000/
```

In [ ]:
import zipfile, pathlib, os, glob

EXTRACT_TO = '/content/dataset_raw'
dzips = sorted(glob.glob(os.path.join(BASE, 'datasets', '*.zip')), key=os.path.getmtime)
assert dzips, 'لا يوجد ملف zip في datasets/.'
if not os.path.exists(EXTRACT_TO):
    print('جارٍ فكّ ضغط البيانات ...', dzips[-1])
    with zipfile.ZipFile(dzips[-1]) as z:
        z.extractall(EXTRACT_TO)
    print('تم.')
else:
    print('مفكوك مسبقاً.')

EXPECTED = {'other', 'y50', 'y100_new', 'y100_old', 'y200_new', 'y200_old', 'y250', 'y500', 'y1000'}

def is_class_dir(p):
    try:
        subs = {s.name for s in p.iterdir() if s.is_dir()}
    except Exception:
        return False
    return len(EXPECTED & subs) >= 4

DATASET_DIR = None
if is_class_dir(pathlib.Path(EXTRACT_TO)):
    DATASET_DIR = EXTRACT_TO
else:
    for d in pathlib.Path(EXTRACT_TO).rglob('*'):
        if d.is_dir() and is_class_dir(d):
            DATASET_DIR = str(d)
            break

print('DATASET_DIR =', DATASET_DIR)
assert DATASET_DIR, 'لم أجد مجلد الفئات داخل الـzip.'

In [ ]:
import pathlib
EXPECTED = {'other', 'y50', 'y100_new', 'y100_old', 'y200_new', 'y200_old', 'y250', 'y500', 'y1000'}
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
root = pathlib.Path(DATASET_DIR)
assert root.is_dir(), 'المجلد غير موجود: ' + str(DATASET_DIR)
found = {d.name for d in root.iterdir() if d.is_dir()}
total = 0
for c in sorted(found):
    n = len([p for p in (root / c).iterdir() if p.suffix.lower() in EXTS])
    total += n
    print(c, '->', n, '(قليل جداً — أضف المزيد)' if n < 30 else '')
print('الإجمالي:', total)
missing = EXPECTED - found
extra = found - EXPECTED
if missing:
    print('فئات ناقصة:', sorted(missing))
if extra:
    print('مجلدات بأسماء غير متوقعة:', sorted(extra))
if not missing and not extra:
    print('بنية المجلدات صحيحة.')

## ٤) التكثير + التقسيم الآمن (بلا تسريب) — **متوازن**

يقسّم الصور الأصلية إلى تدريب/تحقق **قبل** التكثير، ويُبقي صور التحقق خاماً.

الوضع **المتوازن** (`--target_per_class 3200`): يُعطي الفئات النادرة (مثل `y100_new=170`) عدد نسخ أكبر تلقائياً حتى تتساوى الفئات تقريباً — بدل تثبيت 12 نسخة للجميع الذي كان يُبقي اختلال التوازن. راقب السطر المطبوع لكل فئة للتأكّد أن الأعداد تقاربت (~3200 لكل فئة).

In [ ]:
!python tools/augment_dataset.py --input_dir "{DATASET_DIR}" --output_dir /content/dataset_aug --target_per_class 3200 --val_split 0.2

## ٥) التدريب

قد يستغرق 20–60 دقيقة حسب حجم بياناتك. التدريب الآن يراقب `val_loss` (أصدق من الدقّة على بيانات غير متوازنة) ويختار أفضل مرحلة على أساسه.

لتجربة دقة أعلى (أبطأ): غيّر `mobilenetv2` إلى `efficientnetb0`.

In [ ]:
!python tools/train_bigash_model.py --data_dir /content/dataset_aug --output_dir /content/out --backbone mobilenetv2

## ٦) مراجعة المخرجات

السكربت يطبع الدقة النهائية ومصفوفة الالتباس أعلاه. الخلية التالية تتأكد من إنشاء الملفات.

In [ ]:
import pathlib
out = pathlib.Path('/content/out')
for f in ['bigash_v18.tflite', 'labels.txt']:
    p = out / f
    print(f, '->', str(p.stat().st_size // 1024) + ' KB' if p.exists() else 'مفقود!')
print()
print('labels.txt:')
print(pathlib.Path('/content/out/labels.txt').read_text(encoding='utf-8'))

## ٦٫٥) تقييم عبر مسار التطبيق + التأكد من رفض الـ5 روبية (مهم قبل العملاء)

هذه الخطوة تقيس ما سيختبره المستخدم فعلاً، لا مجرد `val_accuracy`:
- تُمرّر صور اختبار حقيقية عبر **نفس** مسار التطبيق (خام [0,255]، قص مركزي، تدوير رباعي TTA، ثم بوابات القرار 0.35/0.10/0.15/0.12).
- تتأكّد أن **الـ5 روبية وأي عملة أجنبية تُرفض** (لا تُعلَن كفئة يمنية) — وإلا يفشل السكربت ويطلب عدم الشحن.
- تطبع لكل فئة يمنية نسبة الصور التي تجاوزت البوابات (نسبة القراءة الواقعية) ومتوسط الثقة.

جهّز مجلد `test_set/` على Drive: مجلد فرعي لكل فئة، وضع صور الـ5 روبية والعملات الأجنبية تحت `test_set/other/`.

In [ ]:
import os
# مجلد اختبار (مُوصى به قبل العملاء): ضعه في Drive باسم datasets/test_set
# فيه مجلد فرعي لكل فئة، وأهمها other/ يحوي صور الـ5 روبية والعملات الأجنبية.
TEST_DIR = os.path.join(BASE, 'datasets', 'test_set')  # ✏️ عدّله إن لزم
if os.path.isdir(TEST_DIR):
    !python tools/eval_model.py --model /content/out/bigash_v18.tflite --labels /content/out/labels.txt --test_dir "{TEST_DIR}"
else:
    print('⚠️ لا يوجد مجلد اختبار في', TEST_DIR)
    print('   أنشئ test_set/ (مجلد لكل فئة + other/ فيه الـ5 روبية) لتشغيل التقييم — مُوصى به قبل العملاء.')

## ٧) التنزيل ووضع الملفات في التطبيق

In [ ]:
import shutil, pathlib
dst = pathlib.Path(BASE) / 'runs'
dst.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/out/bigash_v18.tflite', dst / 'bigash_v18.tflite')
shutil.copy('/content/out/labels.txt', dst / 'labels.txt')
print('نُسخت المخرجات إلى:', dst)

from google.colab import files
files.download('/content/out/bigash_v18.tflite')
files.download('/content/out/labels.txt')

### وضع الملفات في التطبيق

1. انسخ `bigash_v18.tflite` و `labels.txt` إلى مجلد `assets/` (استبدل القديمة).
2. شغّل: `flutter clean` ثم `flutter pub get` ثم `flutter build apk --release`.
3. اختبر على عملات حقيقية بإضاءات وزوايا وخلفيات مختلفة.

### إذا بقيت الدقة ضعيفة

- `val_accuracy` منخفض ⇐ البيانات غير كافية: أضف صوراً حقيقية أكثر لكل فئة (هذا أهم عامل).
- راجع مصفوفة الالتباس لمعرفة أي فئتين يخلط النموذج بينهما، وزِد صورهما.